In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configurations

In [2]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    feature_cols = ["symbol_id", "time_id"] \
        + [f"feature_{idx:02d}" for idx in range(79)] \
        + [f"responder_{idx}_lag_1" for idx in range(9)]
    categorical_cols = []

# Load Data

In [3]:
train = pl.scan_parquet("train.parquet").collect().to_pandas()
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
train.shape, valid.shape

((21022056, 104), (1082224, 104))

In [4]:
# Trick of boosting LB score: 0.45->0.49
# train = pd.concat([train, valid]).reset_index(drop=True)
# train.shape

# GBDT models

In [5]:
def get_model(seed):
    # XGBoost parameters
    XGB_Params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 200,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 1,
        'reg_lambda': 5,
        'random_state': seed,
        'tree_method': 'gpu_hist',
        'device' : 'cuda',
        'n_gpus' : 2,
    }
    
    XGB_Model = XGBRegressor(**XGB_Params)
    return XGB_Model

# Training model

In [6]:
X_train = train[ CONFIG.feature_cols ]
y_train = train[ CONFIG.target_col ]
w_train = train[ "weight" ]
X_valid = valid[ CONFIG.feature_cols ]
y_valid = valid[ CONFIG.target_col ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

((21022056, 90),
 (21022056,),
 (21022056,),
 (1082224, 90),
 (1082224,),
 (1082224,))

In [7]:
%%time
model = get_model(CONFIG.seed)
model.fit( X_train, y_train, sample_weight=w_train)

CPU times: user 10min 50s, sys: 6.9 s, total: 10min 57s
Wall time: 1min 14s


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_gpus=2, n_jobs=None,
             num_parallel_tree=None, ...)

In [19]:
importances = model.feature_importances_

importance_df = pd.DataFrame({
    "feature":  CONFIG.feature_cols,
    "importance": importances
})

# Sort features by importance descending
importance_df.sort_values("importance", ascending=False, inplace=True)
importance_df.reset_index(drop=True, inplace=True)
importance_df

,feature,importance
0,feature_75,0.032681
1,feature_36,0.029503
2,symbol_id,0.028824
3,feature_06,0.026378
4,feature_52,0.019835
...,...,...
85,feature_03,0.005279
86,feature_41,0.005212
87,feature_71,0.005038
88,feature_54,0.005021


In [20]:
importance_df[:50]

,feature,importance
0,feature_75,0.032681
1,feature_36,0.029503
2,symbol_id,0.028824
3,feature_06,0.026378
4,feature_52,0.019835
5,feature_04,0.017547
6,feature_61,0.017222
7,responder_2_lag_1,0.016921
8,feature_76,0.016823
9,feature_66,0.016750


In [8]:
y_pred_train1 = model.predict(X_train.iloc[:X_train.shape[0]//2])
y_pred_train2 = model.predict(X_train.iloc[X_train.shape[0]//2:])
train_score = r2_score(y_train, np.concatenate([y_pred_train1, y_pred_train2], axis=0), sample_weight=w_train )
train_score

0.03463560342788696

In [9]:
y_pred_valid = model.predict(X_valid)
valid_score = r2_score(y_valid, y_pred_valid, sample_weight=w_valid )
valid_score

0.005550682544708252

# Compare with downloaded js24-trained-gbdt-model (Motono)

In [10]:
model_path = "motono_xgb.pkl"
with open( model_path, "rb") as fp:
    result = pickle.load(fp)
    xgb_model_motono = result["model"]

xgb_feature_cols = ["symbol_id", "time_id"] + CONFIG.feature_cols

y_pred_valid_xgb = xgb_model_motono.predict(X_valid)
valid_score = r2_score( y_valid, y_pred_valid_xgb, sample_weight=w_valid )
print('xgb Motono', valid_score)

xgb Motono 0.011727094650268555


# Load js-with-lags-trained-xgb (xiang sheng)

In [11]:
model_path = "xiangsheng_xgb.pkl"
with open( model_path, "rb") as fp:
    result = pickle.load(fp)
    xgb_model_xiangsheng = result["model"]

y_pred_valid_xgb = xgb_model_xiangsheng.predict(X_valid)
valid_score = r2_score( y_valid, y_pred_valid_xgb, sample_weight=w_valid )
print('xgb xiang sheng', valid_score)

xgb xiang sheng 0.01171940565109253


# CV by symbol_id

In [12]:
cross_validation = False
if cross_validation:    
    y_means = { symbol_id : -1 for symbol_id in range(39) }
    for symbol_id, gdf in train[["symbol_id", CONFIG.target_col]].groupby("symbol_id"):
        y_mean = gdf[ CONFIG.target_col ].mean()
        y_means[symbol_id] = y_mean
        print(f"symbol_id = {symbol_id}, y_means = {y_mean:.5f}")

In [13]:
cross_validation = False
if cross_validation:    
    cv_detail = { symbol_id : 0 for symbol_id in range(39) }
    for symbol_id, gdf in valid.groupby("symbol_id"):
        X_valid = gdf[ CONFIG.feature_cols ]
        y_valid = gdf[ CONFIG.target_col ]
        w_valid = gdf[ "weight" ]
        y_pred_valid = model.predict(X_valid)
        score = r2_score(y_valid, y_pred_valid, sample_weight=w_valid )
        cv_detail[symbol_id] = score
        
        print(f"symbol_id = {symbol_id}, score = {score:.5f}")

In [14]:
plot_cv = False
if plot_cv: 
    sids = list(cv_detail.keys())
    plt.bar(sids, [cv_detail[sid] for sid in sids])
    plt.grid()
    plt.xlabel("symbol_id")
    plt.ylabel("CV score")
    plt.show()

# Save result

In [15]:
save_model = False
if save_model:
    result = {
        "model" : model,
        "cv" : valid_score,
        "cv_detail" : cv_detail,
        "y_mean" : y_means,
    }
    with open("result.pkl", "wb") as fp:
        pickle.dump(result, fp)